# Test Gemma4 LAN Server

Notebook này test model `gemma4:latest` đang chạy trên máy LAN `192.168.0.29:11434`.

Mặc định dùng API tương thích Ollama.

In [ ]:
import json
import time
import requests

HOST = "192.168.0.29"
PORT = 11434
BASE_URL = f"http://{HOST}:{PORT}"
MODEL = "gemma4:latest"
TIMEOUT = 120

print("BASE_URL:", BASE_URL)
print("MODEL:", MODEL)

## 1. Kiểm tra server còn sống không

In [ ]:
def request_json(method, path, **kwargs):
    url = BASE_URL + path
    start = time.time()
    response = requests.request(method, url, timeout=TIMEOUT, **kwargs)
    elapsed = time.time() - start
    print(f"{method} {url} -> {response.status_code} ({elapsed:.2f}s)")
    response.raise_for_status()
    if response.text.strip():
        return response.json()
    return None

try:
    result = request_json("GET", "/api/tags")
    print(json.dumps(result, ensure_ascii=False, indent=2)[:4000])
except Exception as exc:
    print("Lỗi kết nối hoặc server không phải Ollama API:")
    print(repr(exc))

## 2. Test generate không stream

Dùng `stream=False` để lấy một JSON response duy nhất, dễ debug.

In [ ]:
payload = {
    "model": MODEL,
    "prompt": "Bạn là MeiRobo. Hãy trả lời ngắn gọn bằng tiếng Việt: Meiko Automation làm gì?",
    "stream": False,
    "options": {
        "temperature": 0.2,
        "num_predict": 120
    }
}

try:
    result = request_json("POST", "/api/generate", json=payload)
    print("\n=== RESPONSE ===")
    print(result.get("response", ""))
    print("\n=== RAW JSON ===")
    print(json.dumps(result, ensure_ascii=False, indent=2)[:4000])
except Exception as exc:
    print("Generate failed:", repr(exc))

## 3. Test chat không stream

In [ ]:
payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "system",
            "content": "Bạn tên là MeiRobo. Trả lời ngắn gọn, tự nhiên bằng tiếng Việt, tối đa 2 câu."
        },
        {
            "role": "user",
            "content": "Bạn là ai?"
        }
    ],
    "stream": False,
    "options": {
        "temperature": 0.2,
        "num_predict": 120
    }
}

try:
    result = request_json("POST", "/api/chat", json=payload)
    print("\n=== RESPONSE ===")
    print(result.get("message", {}).get("content", ""))
    print("\n=== RAW JSON ===")
    print(json.dumps(result, ensure_ascii=False, indent=2)[:4000])
except Exception as exc:
    print("Chat failed:", repr(exc))

## 4. Test stream token

In [ ]:
payload = {
    "model": MODEL,
    "prompt": "Viết 3 gạch đầu dòng ngắn về lợi ích của robot trong nhà máy.",
    "stream": True,
    "options": {
        "temperature": 0.2,
        "num_predict": 160
    }
}

url = BASE_URL + "/api/generate"
start = time.time()
first_token_time = None
full_text = []

try:
    with requests.post(url, json=payload, stream=True, timeout=TIMEOUT) as response:
        print("status:", response.status_code)
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line:
                continue
            item = json.loads(line)
            token = item.get("response", "")
            if token and first_token_time is None:
                first_token_time = time.time() - start
                print(f"\nfirst_token_time: {first_token_time:.2f}s\n")
            if token:
                print(token, end="", flush=True)
                full_text.append(token)
            if item.get("done"):
                print(f"\n\ntotal_time: {time.time() - start:.2f}s")
                break
except Exception as exc:
    print("Streaming failed:", repr(exc))

## 5. Hàm tiện ích để gọi lại nhiều lần

In [ ]:
def ask_gemma4(question, system_prompt="Bạn là MeiRobo. Trả lời ngắn gọn bằng tiếng Việt, tối đa 2 câu."):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        "stream": False,
        "options": {
            "temperature": 0.2,
            "num_predict": 160,
        },
    }
    start = time.time()
    result = request_json("POST", "/api/chat", json=payload)
    answer = result.get("message", {}).get("content", "")
    return {
        "question": question,
        "answer": answer,
        "elapsed": time.time() - start,
        "raw": result,
    }

tests = [
    "Xin chào, bạn tên là gì?",
    "Hãy giới thiệu về Meiko Automation.",
    "Bạn có thể nhảy hoặc múa võ không?",
]

for question in tests:
    print("\n---")
    result = ask_gemma4(question)
    print("Q:", result["question"])
    print("A:", result["answer"])
    print(f"elapsed: {result['elapsed']:.2f}s")